Library

In [2]:
import os
import pandas as pd
import pypdf
from pypdf import PdfReader
import pdfplumber
import re
import nltk
from nltk.corpus import stopwords
from nltk.tokenize import word_tokenize


# Load Text Data

In [3]:
from pathlib import Path

try:
    from pypdf import PdfReader
except Exception:
    PdfReader = None

personal_path = r"C:\Users\nhipk\OneDrive\0. Uni2024\Sem4\ML for Economists\FinalProject" #Edit your personal path here
folder_path = os.path.join(personal_path, "TermpaperFilesILIAS", "C_BundesbankMonthlyReports")
pdf_files = list(sorted(Path(folder_path).glob("*.pdf")))


In [20]:
def load_pdfs_from_folder(folder_path):
    """Read all PDF files from a specified folder, skip the first 4 and last 4 pages, and extract text."""
    data = []

    # Check if the target folder exists
    if not os.path.exists(folder_path):
        print(f"Error: The directory '{folder_path}' does not exist.")
        return None

    # Iterate through all items within the folder
    for filename in os.listdir(folder_path):
        # Process only files with a .pdf extension (case-insensitive)
        if filename.lower().endswith(".pdf"):
            file_path = os.path.join(folder_path, filename)
            print(f"Processing: {filename}...")

            try:
                # Initialize the PDF reader
                reader = PdfReader(file_path)
                total_pages = len(reader.pages)
                
                # Safe check: Only process if the file has more than 8 pages
                if total_pages > 8:
                    full_text = ""
                    
                    # Slice the pages to skip the first 4 and last 4 pages
                    target_pages = reader.pages[4:-4]
                    
                    # Loop through the sliced pages to extract text content
                    for page in target_pages:
                        text = page.extract_text()
                        if text:
                            full_text += text + "\n"

                    # Append the filename and extracted text metadata to the list
                    data.append({"file_name": filename, "text": full_text.strip()})
                else:
                    print(f"⚠️ Skipped '{filename}': File only has {total_pages} pages (too short to drop 4 start/end pages).")

            except Exception as e:
                print(f"Error reading file {filename}: {e}")

    # Convert the extracted data into a pandas DataFrame for seamless NLP/Sentiment Analysis
    df = pd.DataFrame(data)
    return df

# Execute the extraction process
df_sentiment = load_pdfs_from_folder(folder_path)

# Display a preview of the resulting DataFrame
if df_sentiment is not None and not df_sentiment.empty:
    print(f"\nSuccessfully loaded {len(df_sentiment)} files!")
    display(df_sentiment)
else:
    print("\nNo PDF files found or the folder is empty.")

Processing: 2009-12-monatsbericht-data.pdf...
Processing: 2010-12-monatsbericht-data.pdf...
Processing: 2011-12-monatsbericht-data.pdf...
Processing: 2012-12-monatsbericht-data.pdf...
Processing: 2013-12-monatsbericht-data.pdf...
Processing: 2014-12-monatsbericht-data.pdf...
Processing: 2015-12-monatsbericht-data.pdf...
Processing: 2016-12-monatsbericht-data.pdf...
Processing: 2017-12-monatsbericht-data.pdf...
Processing: 2018-12-monatsbericht-data.pdf...
Processing: 2019-12-monatsbericht-data.pdf...
Processing: 2020-12-monatsbericht-data.pdf...
Processing: 2021-12-monatsbericht-data.pdf...
Processing: 2022-12-monatsbericht-data.pdf...
Processing: 2023-12-monatsbericht-data.pdf...

Successfully loaded 15 files!


,file_name,text
0,2009-12-monatsbericht-data.pdf,DEUTSCHE\nBUNDESBANK\nMonthly Report\nDecember...
1,2010-12-monatsbericht-data.pdf,DEUTSCHE\nBUNDESBANK\nMonthly Report\nDecember...
2,2011-12-monatsbericht-data.pdf,Commentaries Economic conditions\nUnderlying t...
3,2012-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...
4,2013-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...
5,2014-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...
6,2015-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...
7,2016-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...
8,2017-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...
9,2018-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...


# Process Data

In [21]:
# Add date column as key
df_sentiment["report_date"] = (
    df_sentiment["file_name"].str[:7]
    )
df_sentiment["word_count"] = df_sentiment["text"].str.split().str.len()

display(df_sentiment)


,file_name,text,report_date,word_count
0,2009-12-monatsbericht-data.pdf,DEUTSCHE\nBUNDESBANK\nMonthly Report\nDecember...,2009-12,100371
1,2010-12-monatsbericht-data.pdf,DEUTSCHE\nBUNDESBANK\nMonthly Report\nDecember...,2010-12,102536
2,2011-12-monatsbericht-data.pdf,Commentaries Economic conditions\nUnderlying t...,2011-12,113864
3,2012-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2012-12,119657
4,2013-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2013-12,113182
5,2014-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2014-12,105413
6,2015-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2015-12,112959
7,2016-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2016-12,120586
8,2017-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2017-12,127829
9,2018-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2018-12,116731


In [ ]:
# Stop words and Custom Words
import ssl

try:
    _create_unverified_https_context = ssl._create_unverified_context
except AttributeError:
    pass
else:
    ssl._create_default_https_context = _create_unverified_https_context

nltk.download('stopwords')
nltk.download('punkt')
nltk.download('wordnet')
nltk.download('punkt_tab')

# Load standard English stop words
stop_words = set(stopwords.words("english"))
print(stop_words)


{'or', 'isn', "wouldn't", 'them', 'few', 'doing', "doesn't", 'no', "they've", 'over', 'hers', 'be', 'further', 'those', "i'm", 'myself', 'y', "isn't", 'there', "you've", 'themselves', 'wouldn', 'here', "they'll", "she's", 'below', "that'll", 'haven', 'now', 'so', "we'd", 'some', 'hadn', 'your', 'of', 'this', 'from', 'on', 'who', 'both', 'his', 'than', 'then', 'shouldn', 'was', 'an', "hadn't", 'not', 'which', 'once', 'all', 'needn', 'does', "he'd", 'these', 'being', 'but', 'their', 'been', "i'll", "he'll", 'my', 'as', 'doesn', 't', 'by', 'have', 'don', 'until', 'our', 'too', 'mustn', "we're", 'they', 'again', 's', 'won', 'am', 'if', 'through', "didn't", 'i', 'against', 'its', "haven't", 'other', "she'll", 're', 'for', 'a', 'do', "i've", "hasn't", 'at', 'while', 'yourself', "wasn't", 'same', "we've", 'theirs', "they'd", "it'd", 'own', 'yourselves', "it's", 'o', 'should', 'hasn', 'is', "it'll", "mustn't", 'mightn', 'and', 'we', 'whom', 'down', 'weren', 'were', 've', 'how', "don't", 'shan'

[nltk_data] Downloading package stopwords to
[nltk_data]     C:\Users\nhipk\AppData\Roaming\nltk_data...
[nltk_data]   Package stopwords is already up-to-date!
[nltk_data] Downloading package punkt to
[nltk_data]     C:\Users\nhipk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package wordnet to
[nltk_data]     C:\Users\nhipk\AppData\Roaming\nltk_data...
[nltk_data]   Package wordnet is already up-to-date!
[nltk_data] Downloading package punkt_tab to
[nltk_data]     C:\Users\nhipk\AppData\Roaming\nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


In [ ]:
# A list of custom words to be removed
custom_stop_words = {
    "report", "monthly", "bundesbank", "deutsche", "german", 
    "central", "bank", "european", "euro", "zone", "europe", 
    "ecb", "monetary", "policy"
}
all_stop_words = custom_stop_words.union(stop_words)

# Clean white spaces and convert to lowercase
df_sentiment['cleaned_text'] = (
    df_sentiment['text']
    .str.replace(r'\s+', ' ', regex=True)
    .str.lower()
)

# Remove stop words
df_sentiment['cleaned_text'] = df_sentiment['cleaned_text'].apply(
    lambda x: ' '.join(
        word for word in word_tokenize(x) if word not in all_stop_words
    )
)

# Tokenize cleaned text
df_sentiment['tokens'] = df_sentiment['cleaned_text'].apply(word_tokenize)

df_sentiment["word_count"] = df_sentiment["cleaned_text"].str.split().str.len()
df_sentiment["token_count"] = df_sentiment["tokens"].str.len()

display(df_sentiment)

,file_name,text,report_date,word_count,cleaned_text,tokens,token_count
0,2009-12-monatsbericht-data.pdf,DEUTSCHE\nBUNDESBANK\nMonthly Report\nDecember...,2009-12,89603,december 2009 5 commentaries economic conditio...,"[december, 2009, 5, commentaries, economic, co...",89605
1,2010-12-monatsbericht-data.pdf,DEUTSCHE\nBUNDESBANK\nMonthly Report\nDecember...,2010-12,91902,december 2010 5 commentaries economic conditio...,"[december, 2010, 5, commentaries, economic, co...",91911
2,2011-12-monatsbericht-data.pdf,Commentaries Economic conditions\nUnderlying t...,2011-12,102530,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin...",102534
3,2012-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2012-12,108359,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin...",108366
4,2013-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2013-12,99696,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin...",99701
5,2014-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2014-12,95287,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin...",95294
6,2015-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2015-12,101091,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin...",101099
7,2016-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2016-12,107801,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin...",107813
8,2017-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2017-12,112955,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin...",112965
9,2018-12-monatsbericht-data.pdf,Commentaries\nEconomic conditions\nUnderlying ...,2018-12,104545,commentaries economic conditions underlying tr...,"[commentaries, economic, conditions, underlyin...",104558


# Positive/Negative Dicts

In [33]:
import pysentiment2 as ps

# Initialize the Loughran-McDonald financial dictionary
lm = ps.LM()

positive_words = list(lm._posset)
negative_words = list(lm._negset)

# Print the total counts and a small sample
print(f"Total Positive Words (Stemmed): {len(positive_words)}")
print(f"Sample Positive Words: {sorted(positive_words)[:15]}\n")

print(f"Total Negative Words (Stemmed): {len(negative_words)}")
print(f"Sample Negative Words: {sorted(negative_words)[:15]}")

Total Positive Words (Stemmed): 140
Sample Positive Words: ['abund', 'acclaim', 'accomplish', 'achiev', 'adequ', 'advanc', 'advantag', 'allianc', 'assur', 'attain', 'attract', 'beauti', 'benefici', 'benefit', 'better']

Total Negative Words (Stemmed): 893
Sample Negative Words: ['abandon', 'abdic', 'aberr', 'abet', 'abnorm', 'abolish', 'abrog', 'abrupt', 'abruptli', 'absenc', 'absente', 'abus', 'accid', 'accident', 'accus']


In [34]:
# Helper functions to extract Positive and Negative counts
def get_financial_sentiment(text):
    tokens = lm.tokenize(text)
    score = lm.get_score(tokens)
    
    return score['Positive'], score['Negative'], score['Polarity']

# Apply to DataFrame
# We zip the results into three new columns
df_sentiment['lm_positive'], df_sentiment['lm_negative'], df_sentiment['lm_polarity'] = zip(
    *df_sentiment['cleaned_text'].apply(get_financial_sentiment)
)


display(df_sentiment[['file_name', 'word_count', 'lm_positive', 'lm_negative', 'lm_polarity']])

,file_name,word_count,lm_positive,lm_negative,lm_polarity
0,2009-12-monatsbericht-data.pdf,89603,490,667,-0.152982
1,2010-12-monatsbericht-data.pdf,91902,479,829,-0.267584
2,2011-12-monatsbericht-data.pdf,102530,459,751,-0.241322
3,2012-12-monatsbericht-data.pdf,108359,584,721,-0.104981
4,2013-12-monatsbericht-data.pdf,99696,590,886,-0.200542
5,2014-12-monatsbericht-data.pdf,95287,438,652,-0.196330
6,2015-12-monatsbericht-data.pdf,101091,499,672,-0.147737
7,2016-12-monatsbericht-data.pdf,107801,599,986,-0.244164
8,2017-12-monatsbericht-data.pdf,112955,608,853,-0.167693
9,2018-12-monatsbericht-data.pdf,104545,576,765,-0.140940
